In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 

from misc_utils import concordance_plot,auto_split_range,compound_argsort,single_agg,double_agg

from pathlib import Path

plt.rcParams['image.aspect'] = 'auto'
plt.rcParams['image.interpolation'] = 'none'

# Loading

In [ ]:
# !wget https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE268179
# !wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE268nnn/GSE268179/soft/

In [ ]:
raw_path = "/home/bmb/haxx/working/ceisel_mumm/raws/sc/"
ann_path = "/home/bmb/haxx/working/ceisel_mumm/data/"

In [ ]:
## We need to isolate each set of files into a directory for a scanpy import 

# !mkdir {raw_path}/GSM8287434_RGC_24+48h_ranger
# !mv {raw_path}/GSM8287434_RGC_24+48h_*.gz {raw_path}/GSM8287434_RGC_24+48h_ranger

# !mkdir {raw_path}/GSM8287438_RGC_12+72h_ranger
# !mv {raw_path}/GSM8287438_RGC_12+72h_*.gz {raw_path}/GSM8287438_RGC_12+72h_ranger

In [ ]:
annotations = pd.read_csv(ann_path + "ceisel_adata_metadata.csv")
print(annotations.shape)
annotations.head()

In [ ]:
annotations.loc[annotations['Unnamed: 0'].duplicated()]

In [ ]:
# Annotration barcodes combine file name with barcode, we're stripping it out to match the count files

stripped_annotation_ids = [id.split("_")[2] for id in annotations['Cell.ID']]
stripped_annotation_ids
file_assignment = ["_".join(id.split("_")[:2]) for id in annotations['Cell.ID']]
annotations['Stripped.Cell.ID'] = stripped_annotation_ids
annotations['File.Assignment'] = file_assignment
annotations.set_index('Stripped.Cell.ID',inplace=True)

middle_annotations = annotations[annotations['File.Assignment'] == '24_48']
outer_annotations = annotations[annotations['File.Assignment'] == '12_72']
middle_annotations.head()
outer_annotations.head()


In [ ]:
# After stripping there are duplicate codes. This is theoretically fine since we can join each frame individually. 
annotations.iloc[annotations.index.duplicated()]

In [ ]:
middle_raw = sc.read_10x_mtx(
    raw_path + "/GSM8287434_RGC_24+48h_ranger",
    prefix="GSM8287434_RGC_24+48h_"
)
print(middle_raw.shape)
outer_raw = sc.read_10x_mtx(
    raw_path + "/GSM8287438_RGC_12+72h_ranger",
    prefix="GSM8287438_RGC_12+72h_"
)
print(outer_raw.shape)

In [ ]:
# Not all barcodes that are in the annotation file are also present in the raw file 
# Presumably annotation was created after original filtering

middle_ann_names = set(middle_annotations.index)
matching_middle = [id in middle_ann_names for id in middle_raw.obs_names]
print(f"middle: {np.sum(matching_middle)} out of {middle_raw.shape[0]}")
middle_raw.obs['annotated'] = matching_middle
outer_ann_names = set(outer_annotations.index)
matching_outer = [id in outer_ann_names for id in outer_raw.obs_names]
print(f"outer: {np.sum(matching_outer)} out of {outer_raw.shape[0]}")
outer_raw.obs['annotated'] = matching_outer


In [ ]:
# Retain only cells that are present in the annotation file
# Then join the annotations

middle_annotated = middle_raw[middle_raw.obs['annotated']]
outer_annotated = outer_raw[outer_raw.obs['annotated']]
middle_annotated.obs = middle_annotated.obs.join(middle_annotations)
outer_annotated.obs = outer_annotated.obs.join(outer_annotations)
middle_annotated.obs.head()
outer_annotated.obs.head()


In [ ]:
joint_raw = ad.concat([middle_annotated,outer_annotated])
joint_raw.shape

In [ ]:
# Dedupes the barcodes that are duplicated 
joint_raw.obs_names_make_unique()

In [ ]:
sc.write(ann_path + "joint_raw.h5ad",joint_raw)

# QC On Annotated Data

In [ ]:
ann_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
joint_raw = sc.read_h5ad(ann_path + "joint_raw_new.h5ad")
joint_raw.shape

In [ ]:
sc.pp.calculate_qc_metrics(joint_raw, inplace=True)

In [ ]:
split_names = [n.split("-") for n in joint_raw.obs_names]
suffix = [n[-1] for n in split_names]

joint_raw.obs["library"] = suffix

plt.figure()
plt.title("Total Counts (All)")
plt.hist(joint_raw.obs["total_counts"],bins=np.arange(0,10000,100))
plt.xlabel("Counts Per Cell")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.title("Cells Per Library")
plt.hist(suffix)
plt.xlabel("Library")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.title("Total Counts Split by Library")
for suffix in set(suffix):
    masked = joint_raw[joint_raw.obs["library"] == suffix]
    plt.hist(masked.obs["total_counts"],bins=np.arange(800,4000,100),alpha=0.3,label=suffix)
plt.xlabel("Counts Per Cell")
plt.ylabel("Frequency")
plt.legend()
plt.show()

# Retaining Signature Genes

#### Parthanatos

In [ ]:
parthanatos_signature_genes = ["Il1b","Aif1","App","Dlg4","Endog","F2","Grin2b","Icam1","Il1b","Map2k1","Mapk1","Mmp9","Parg","Parp1","Plat","S100b","Sirt1"]
parthanatos_signature_genes = [g.lower() for g in parthanatos_signature_genes]
parthanatos_signature_set = set(parthanatos_signature_genes)
parthanatos_signature_set

In [ ]:
parthanatos_present = [p for p in parthanatos_signature_set if p in set(joint_raw.var_names)]
parthanatos_missing = [p for p in parthanatos_signature_set if p not in set(joint_raw.var_names)]

print(parthanatos_present)
print(parthanatos_missing)


parthanatos_replacement_addons = [
    'parga','pargl',
    'dlg4a','dlg4b','dlg4b-1',
    'aif1l',
    'appa','appb',
    'grin2bb'
]

# F2 and s100b are present in the ann, but fall below even very conservative filtering thresholds. Removing. 
parthanatos_present.remove('f2')
parthanatos_present.remove('s100b')

print(joint_raw[:,'f2'].var)
print(joint_raw[:,'s100b'].var)

In [ ]:
# [g for g in joint_raw.var_names if "f2" in g[:3]]

# icam1: searched for the cd54 alias, searched or all icams and found only icam3
# s100b: questionably present, difficult to analogize? Letter soup present is t,a,w,v,u, and z
# f2 should theoretically be present? Not sure why it's not showing up. Omitting 

parthanatos_translated_signature = parthanatos_present + parthanatos_replacement_addons
parthanatos_translated_signature


#### Apoptosis

In [ ]:
apoptosis_signature_genes = ["Bax","Cycs","Akt1","Akt1","Bax","Bcl2l1","Birc2","Cycs","Ikbkb","Irak1","Pik3ca","Pik3r1","Pik3r2","Ppp3ca","Ppp3cb","Ppp3r1","Prkaca","Prkacb","Prkar1a","Prkar1b","Xiap"]
apoptosis_signature_genes = [g.lower() for g in apoptosis_signature_genes]
apoptosis_signature_set = set(apoptosis_signature_genes)

apoptosis_present = [p for p in apoptosis_signature_set if p in set(joint_raw.var_names)]
apoptosis_missing = [p for p in apoptosis_signature_set if p not in set(joint_raw.var_names)]

print(apoptosis_present)
print(apoptosis_missing)

In [ ]:
# Same as above, cycsa is gettinf filtered out, so I'm omitting. 
apoptosis_replacement_addons = [
    'baxb','baxa','baxa-1',
    'cycsb', # 'cycsa',
    'prkacaa','prkacab',
    'prkacba','prkacbb',
    'prkar1aa','prkar1ab','prkar1b',
    'ppp3r1b', 'ppp3r1a'
]

In [ ]:
# Manual replacement search 
print([g for g in joint_raw.var_names if "birc2" in g])

# baxa,baxb,baxa-1 for bax

apoptosis_translated_signature = apoptosis_present + apoptosis_replacement_addons
apoptosis_translated_signature


#### Necroptosis General(?)

In [ ]:
# These duplicates are an exact copy from the xls from the paper. Why? I don't know. The coefficients they claim appear duplicated so I don't think it's affecting their analysis?
necroptosis_signature_genes = ["Pgam5","Ripk3","Hmgb1","Dnm1l","Hmgb1","Mlkl","Pgam5","Ripk1","Ripk3","Mlkl","Ripk1"] 
necroptosis_signature_genes = [g.lower() for g in necroptosis_signature_genes]
necroptosis_signature_set = set(necroptosis_signature_genes)

necroptosis_present = [p for p in necroptosis_signature_set if p in set(joint_raw.var_names)]
necroptosis_missing = [p for p in necroptosis_signature_set if p not in set(joint_raw.var_names)]

print(necroptosis_present)
print(necroptosis_missing)


In [ ]:
print([g for g in joint_raw.var_names if "flj" in g])

# Mlkl appears wholly missing. The only alias I see online is flj34389, which is also absent (very few fljs generally)
necroptosis_replacement_addons = [
    'hmgb1a','hmgb1b',
    'ripk1l',
]

# HMGB1A & B are gregarious in development, we can mask them and see what happens

necroptosis_translated_signature = necroptosis_present + necroptosis_replacement_addons
necroptosis_translated_signature

In [ ]:
# necroptosis_translated_signature = ['pgam5', 'ripk3', 'dnm1l', 'ripk1l'] # Kicking out the dev

# Hand-curated instead, attr Ceisel
necroptosis_translated_signature = [
    'ripk1l','ripk3','fadd',
    'cyldb','cylda','cyldl',
    'igfbp2a','igfbp2b',
    'pgam5','tnfrsf1a','vdac1','vdac2','rbck1','tradd','dnm1l']

### Populations

In [ ]:

population_marker_sets = {
    'cones': ['gnat2','arr3a','arr3b'],
    'rods': ['rho','gnat1','gngt1','sagb','pde6a','pde6b','nrl'],
    'muller_glia':['rlbp1a','rx1','crabp1a','gpr37a',],
    'rgcs':['rbpms2b','isl2b','uchl1','rbpms2a','pou4f2'],
    'rgc_secondary':['nrn1a','isl2b','islr2',],
    'microglia':['mfap4','mpeg1.1','csf1ra','spi1b'],
    'bipolar_cells':['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',], 
    'amacrine_cells':['pax6b','slc6a1b','pax10','nhlh2','pvalb6'], # pax6a got filtered
    'horizontal_cells':['onecutl','tnr','rem1','CR361564.1','cx52.9','ompa','opn9'], # onecut ambiguous, cx55.5, slc4a5b seem fully missing from the index
    'undifferentiated':['sox2','olig2','hes6','hmgn2','hmga1a','atoh7'],
    'pr_precursors':['tulp1a','opn6a','tmem244'],
    'erythrocytes':['hbbe2','prdx2','creg1'],
    # 'yama':['pou5f3','myca','mycb','klf17','sox2'],
}

all_cell_markers = [g for l in population_marker_sets.values() for g in l]
all_cell_markers

In [ ]:
for population in population_marker_sets:
    with open(f"population_markers.txt","a") as f:
        f.write(f"{population}: {str(",".join(population_marker_sets[population]))}\n")


In [ ]:
candidates = {}
for marker in all_cell_markers:
    candidates[marker] = [g for g in joint_raw.var_names if marker in g]
candidates

# Filtering For Salience

## Finalized

In [ ]:
keepers = {
    'parthanatos':[
        'mapk1', 'aif1l', 'dlg4a', 'appa', 'grin2bb', 'appb', 'parp1', 'plat',
        'endog', 'mmp9', 'parga', 'sirt1', 'map2k1', 'dlg4b', 'dlg4b-1', 'il1b','pargl'],
    'necroptosis':[
        'ripk3', 'tradd', 'cylda', 'fadd', 'pgam5', 'igfbp2a', 'cyldl',
        'ripk1l', 'igfbp2b', 'tnfrsf1a', 'vdac2', 'vdac1', 'cyldb', 'rbck1','dnm1l'],
    'apoptosis':[
        'pik3r1', 'prkar1b', 'baxb', 'prkar1aa', 'baxa', 'baxa-1', 'prkacaa',
        'cycsb', 'prkacbb', 'pik3r2', 'ppp3r1b', 'prkacab', 'ikbkb', 'irak1',
        'akt1', 'ppp3cb', 'xiap', 'ppp3r1a', 'prkar1ab', 'bcl2l1', 'birc2','ppp3ca', 'prkacba', 'pik3ca'],
    **population_marker_sets
}

In [ ]:
sc.pp.filter_genes(joint_raw, min_cells=1)
sc.pp.filter_cells(joint_raw, min_counts=1)

copy = joint_raw.copy()
sc.pp.log1p(copy)
sc.pp.highly_variable_genes(copy, n_top_genes=5000)

flat_keepers = set([g for v in keepers.values() for g in v]) # Flatten the above list
keeper_mask = np.zeros(copy.shape[1],dtype=bool)
keeper_mask[copy.var_names.isin(flat_keepers)] = True
# copy.var.loc[flat_keepers,'highly_variable'] = True

joint = joint_raw[:,copy.var["highly_variable"] | keeper_mask].copy()

# Let's annotate the genes per signature for the actual signatures
joint.var['parthanatos'] = [g in keepers['parthanatos'] for g in joint.var_names]
joint.var['necroptosis'] = [g in keepers['necroptosis'] for g in joint.var_names]
joint.var['apoptosis'] = [g in keepers['apoptosis'] for g in joint.var_names]


# sc.pp.downsample_counts(joint,counts_per_cell=1000)
sc.pp.normalize_total(joint,exclude_highly_expressed=True)
# sc.pp.normalize_total(joint)
joint.layers['raw'] = joint.X.copy()
sc.pp.log1p(joint)

# Linear batch regression?
# Unfortunately linearly regressing out batch (or any other batch resolution idea) is probably bad because it would nuke a dangerouly large amount of the time variance. =/
# sc.pp.regress_out(joint, ['Batch'])

In [ ]:
sc.write(data_path + "joint_annotated_filtered.h5ad",joint)

# PCA And UMAP

In [ ]:
data_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
joint = sc.read_h5ad(data_path + "joint_annotated_filtered.h5ad")


In [ ]:
print("Computing PCA")
sc.pp.pca(joint, n_comps=50)
print("Computing neighbors")
sc.pp.neighbors(joint)
print("Computing UMAP")
sc.tl.umap(joint)
sc.pl.umap(joint)


In [ ]:
log_total_counts = np.log1p(joint.obs['total_counts'])
joint.obs['log_total_counts'] = log_total_counts
sc.pl.umap(joint, color=["log_total_counts"])
sc.pl.umap(joint, color=["library"])
sc.pl.umap(joint, color=["Batch"])

In [ ]:
time = [c.split("_")[0].strip("h") for c in joint.obs['Condition']]
condition = ["_".join(c.split("_")[1:]) for c in joint.obs['Condition']]
joint.obs['exp_condition'] = condition
joint.obs['time'] = np.array(time).astype(int)


In [ ]:
sc.pl.umap(joint, color=["time"])

In [ ]:
plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [joint.obs['log_total_counts'][joint.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()

In [ ]:
data_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
sc.write(data_path + "joint_annotated_neighbors.h5ad",joint)


# Cell Type Annotation

In [ ]:
data_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data = sc.read_h5ad(data_path + "joint_annotated_neighbors.h5ad")

In [ ]:
sc.tl.leiden(data,resolution=1.5)
sc.pl.umap(data, color=['leiden'],legend_loc='on data')

In [ ]:
# As before
population_marker_sets = {
    'cones': ['gnat2','arr3a','arr3b'],
    'rods': ['rho','gnat1','gngt1','sagb','pde6a','pde6b','nrl'],
    'muller_glia':['rlbp1a','rx1','crabp1a','gpr37a',],
    'rgcs':['rbpms2b','isl2b','uchl1','rbpms2a','pou4f2'],
    'rgc_secondary':['nrn1a','isl2b','islr2',],
    'microglia':['mfap4','mpeg1.1','csf1ra','spi1b'],
    'bipolar_cells':['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',], 
    'amacrine_cells':['pax6b','slc6a1b','pax10','nhlh2','pvalb6'], # pax6a got filtered
    'horizontal_cells':['onecutl','tnr','rem1','CR361564.1','cx52.9','ompa','opn9'], # onecut ambiguous, cx55.5, slc4a5b seem fully missing from the index
    'undifferentiated':['sox2','olig2','hes6','hmgn2','hmga1a','atoh7'],
    'pr_precursors':['tulp1a','opn6a','tmem244'],
    'erythrocytes':['hbbe2','prdx2','creg1'],
    # 'yama':['pou5f3','myca','mycb','klf17','sox2']
}


In [ ]:
sc.pl.dotplot(data,population_marker_sets,groupby="leiden",figsize=(10,8))

In [ ]:
np.sum(data.obs['leiden'] == "28")

In [ ]:
mapping = {
    k:k for k in data.obs['leiden'].unique()
}

# THIS MAPPING WILL SHIFT PER EVALUATION OF LEIDEN, BECAUSE LEIDEN IS STOCHASTIC. RE-DO PER RUN USING THE DOTPLOT


reverse_mapping = {
#     'Bipolar cells':[0,1,2,3,7,10,11,19,],
#     'Horizontal cells':[5,17,],
#     'Cones':[13,18,23,],
#     'Rods':[24],
#     'Muller glia':[8],
    'RGCs':[28],
#     'Microglia':[28],
#     'Amacrine cells':[4,12,21],
#     'Erythrocytes':[27,],
#     'injury':[31,],
}

# reverse_mapping = {
#     'Bipolar cells':[0,2,3,4,7,11,12,14,19,],
#     'Horizontal cells':[1,30],
#     'Cones':[5,10,18,],
#     'Rods':[24,],
#     'Muller glia':[9,],
#     'RGCs':[21,],
#     'Microglia':[29,],
#     'Amacrine cells':[6,15,20,],
#     'Erythrocytes':[27,],
#     'injury':[31,],
# }

mapping |= {
    str(v):k for k,l in reverse_mapping.items() for v in l
} 



# mapping
# Create a new column with renamed clusters
data.obs['cell_type'] = data.obs['leiden'].map(mapping).astype('category')


In [ ]:
sc.pl.dotplot(data,population_marker_sets,groupby="cell_type",figsize=(10,8))

In [ ]:
sc.pl.umap(data,color='cell_type',legend_loc='on data')

In [ ]:
sc.tl.rank_genes_groups(data, 'cell_type')
sc.pl.rank_genes_groups(data, n_genes=25)

In [ ]:
sc.write(data_path + "full_annotations_leiden_new.h5ad",data)

In [ ]:
data.shape

In [ ]:
old_rgcs = [
    99,581,585,628,872,1375,1395,1413,1436,1508,1665,1680,1697,1701,2047,2237,2257,2373,2454,2457,2535,2629,2755,3115,3292,3340,3370,3470,3555,3630,3637,3805,3819,3821,4521,4668,4921,5120,5264,5363,5383,5469,5635,5742,5779,5905,6452,6510,6605,6619,6767,6893,6933,7015,7021,7034,7041,7297,7315,7320,7564,7787,7829,7859,8118,8295,8659,8660,8735,8944,9113,9334,9418,9438,9500,9680,10084,10643,10697,10734,10974,11073,11716,12004,12262,12339,12654,13113,13174,13384,13465,13859,13913,13997,14306,14863,15114,15193,15454,15767,15833,15904,15947,15989,16061,16234,16447,16542,16557,16844,16883,17455,17537,17883,17954,19298,20498,20626,21288,21434,21724,25199,25645,26799,28296,31901,32233,32764,33364,33438,34275,36656,39149,39643,40378,41115,41187,42440,42694,42830,43621,46954,47559,48061,48067,48200,50117,54030,55162,56844,57004,57062,57109,57110,57139,57151,57155,57165,57195,57259,57282,57344,57357,57379,57397,57424,57443,57464,57469,57482,57513,57521,57533,57539,57543,57554,57584,57656,57668,57679,57749,57783,57879,57906,57963,57967,58007,58015,58056,58088,58110,58111,58154,58210,58215,58225,58228,58243,58254,58270,58295,58308,58358,58397,58404,58412,58462,58464,58477,58560,58577,58596,58599,58609,58767,58794,58795,58860,58879,58893,58901,58911,58985,59074,59075,59136,59243,59324,59355,59471,59509,59516,59527,59532,59535,59642,59701,59705,59714,59723,59780,59786,59799,59826,59830,59883,59886,59889,60010,60073,60092,60117,60238,60289,60376,60386,60421,60422,60522,60561,60583,60585,60594,60652,60668,60692,60698,60764,60865,60868,60870,61041,61094,61153,61155,61195,61470,61523,61620,61721,61725,61752,61823,61827,61917,61928,61952,61969,61978,61982,62052,62074,62088,62096,62234,62253,62288,62313,62404,62423,62450,62664,62671,62682,62761,62783,62817,62878,62899,62949,62959,63084,63139,63189,63194,63209,63213,63300,63311,63330,63340,63353,63361,63409,63413,63496,63532,63537,63645,63672,63722,63758,63791,63808,63834,63837,63903,63934,63937,63987,64036,64052,64080,64114,64154,64167,64203,64245,64264,64276,64286,64318,64469,64572,64643,64644,64645,64677,64754,64759,64772,64801,64927,64952,65047,65103,65105,65150,65218,65227,65239,65287,65310,65316,65321,65329,65378,65432,65481,65499,65532,65534,65626,65638,65655,65683,65833,65958,65975,66005,66006,66034,66091,66138,66145,66146,66161,66179,66202,66285,66318,66327,66331,66337,66377,66394,66435,66459,66512,66516,66586,66593,66659,66685,66686,66702,66708,66789,66871,66912,66947,66956,66983,67037,67080,67135,67136,67329,67376,67388,67418,67445,67467,67488,67532,67556,67563,67580,67584,67621,67736,67757,67770,67783,67811,67812,67834,67845,68016,68036,68042,68082,68150,68184,68197,68202,68231,68237,68300,68317,68327,68393,68425,68479,68632,68687,68739,68812,68870,68946,68964,69154,69196,69205,69228,69241,69310,69328,69365,69382,69389,69391,69422,69429,69481,69507,69513,69520,69545,69594,69604,69610,69620,69677,69689,69747,69762,69929,69978,70013,70032,70039,70093,70119,70242,70252,70254,70342,70348,70378,70449,70457,70461,70463,70488,70496,70500,70520,70532,70641,70646,70667,70691,70779,70807,70953,70982,71012,71015,71030,71123,71168,71246,71317,71370,71422,71442,71450,71485,71523,71529,71558,71563,71584,71857,71895,72000,72022,72061,72063,72092,72099,72119,72140,72219,72224,72239,72273,72294,72318,72328,72367,72368,72380,72399,72441,72501,72550,72623,72698,72704,72754,72773,72795,72800,72873,72999,73009,73055,73087,73170,73215,73303,73371,73464,73476,73501,73506,73519,73528,73542,73594,73632,73639,73672,73705,73712,73750,73792,73819,73894,73942,73968,73993,73994,73999,74128,74136,74182,74196,74232,74313,74318,74333,74349,74377,74510,74578,74589,74597,74648,74686,74755,74772,74790,74817,74856,74862,74944,75011,75034,75096,75119,75182,75187,75215,75245,75274,75294,75298,75411,75414,75444,75468,75487,75497,75623,75627,75635,75665,75676,75697,75724,75734,75785,75987,76058,76069,76109,76113,76159,76197,76207,76236,76304,76333,76432,76563,76580,76618,76634,76665,76735,76827,76857,76886,76933,76972,76999,77187,77219,77223,77243,77314,77323,77336,77366,77369,77370,77435,77441,77444,77451,77460,77463,77584,77610,77637,77653,77764,77779,77790,77837,77838,77860,77876,77878,77911,77919,77935,77950,77967,78049,78072,78081,78101,78111,78149,78188,78208,78228,78231,78252,78260,78274,78298,78317,78322,78369,78435,78458,78502,78517,78531,78578,78621,78686,78740,78812,78815,78839,78843,78863,78888,78920,78936,79041,79058,79138,79160,79187,79299,79350,79432,79473,79551,79599,79624,79647,79649,79659,79667,79769,79831,79864,79869,79880,79929,79930,79935,80053,80074,80079,80108,80135,80148,80168,80235,80294,80333,80343,80502,80530,80538,80571,80583,80601,80638,80672,80707,80720,80778,80796,80834,80930,80949,80951,81011,81024,81115,81135,81255,81328,81334,81346,81364,81407,81510,81547,81728,81819,81832,81896,81907,81925,81945,81973,82004,82044,82162,82175,82202,82203,82251,82287,82353,82386,82388,82395,82466,82484,82493,82532,82562,82572,82587,82598,82608,82665,82668,82780,82831,82926,82966,82969,83194,83212,83226,83247,83302,83320,83329,83471,83574,83585,83604,83648,83666,83718,83746,83775,83816,83847,83865,83875,83880,83904,83913,83986,84051,84059,84068,84102,84129,84139,84150,84191,84256,84344,84391,84404,84446,84477,84492,84547,84607,84685,84723,84734,84894,84911,84957,85005,85079,85138,85166,85270,85416,85556,85570,85629,85651,85668,85805,85900,85903,85915,85925,85942,85983,86042,86071,86103,86153,86217,86225,86284,86305,86317,86345,86346,86398,86417,86432,86524,86539,86555,86583,86589,86696,86701,86724,86750,86759,86798,86830,86849,86891,86905,86918,86943,86989,86991,87042,87069,87107,87141,87210,87230,87254,87259,87348,87399,87417,87418,87457,87463,87515,87530,87569,87654,87734,87737,87783,87786,87823,87860,87883,87905,87920,87976,87987,88023,88047,88062,88081,88087,88190,88275,88284,88311,88378,88416,88443,88477,88479,88488,88497,88524,88553,88576,88604,88630,88631,88663,88677,88708,88748,88793,88803,88898,88906,88980,88991,89194,89196,89234,89245,89292,89296,89435,89476,89509,89510,89516,89581,89616,89638,89714,89733,89762,89764,89803,89807,89845,89887,89942,89948,90086,90144,90147,90208,90293,90340,90348,90389,90401,90406,90411,90438,90449,90461,90499,90527,90529,90534,90539,90562,90635,90748,90775,90867,90889,90957,91000,91001,91020,91041,91119,91152,91172,91191,91211,91242,91364,91367,91456,91475,91519,91638,91680,91780,91803,91829,91956,91960,91972,91984,92096,92124,92142,92150,92252,92277,92294,92309,92411,92445,92474,92503,92866,92955,92978,92990,93080,93339,93393,93497,93581,93673,93674,93695,93713,93867,93869,93982,94095,94476,94588,94649,94720,94728,94844,95111,95296,95306,95397,95425,95555,95650,95659,95834,95862,95899,96192,96206,96212,96267,96280,96460,96721,96768,96822,96906,96977,96978,97085,97107,97181,97194,97217,97229,97263,97317,97797,97888,97889,97897,97958,98016,98052,98080,98097,98138,98193,98335,98475,98497,98500,98507,98608,98970,99091,99133,99196,99256,99320,99372,99487,99531,99666,99684,99725,99759,99777,99783,99800,99803,99952,99956,99991,100048,100108,100157,100160,100436,100523,100536,100581,100668,100874,101061,101107,101202,101262,101293,101376,101384,101447,101527,101561,101613,101649,101817,101957,102086,102119,102164,102174,102197,102210,102233,102274,102312,102348,102561,102577,102672,102723,102866,102930,102933,102996,103198,103256,103283,103352,103428,103814,103901,103956,104013,104032,104226,104272,104314,104385,104584,104674,104780,104796,104823,104851,104883,104937,105284,105321,105526,105625,105722,105843,105854,105944,105950,106104,106189,106193,106215,106219,106250,106293,106389,106417,106861,106886,106969,107037,107109,107115,107297,107300,107358,107392,107424,107461,107477,107514,107526,107534,107737,107814,108280,108343,108433,108467,108499,108533,108688,108691,108756,108761,108789,108797,108861,109212,109255,109410,109432,109518,109624,109778,109923,109953,110105,110222,110353,110454,110605,110670,110785,110916,110981,110985,111209,111244,111350,111409,111412,111413,111472,111508,111528,111629,111730,111790,111829,111982,111987,112367,112492,112511,112643,112750,112923,112975,113002,113226,113297,113397,113443,113446,113510,113613,113727,113761,113779,113949,114106,114167,114241,114693,114717,114731,114771,114863,114917,115360,115382,115562,115746,115932,116136,116196,116236,116264,116346,116535,116580,116628,116692,116728,116805,116850,117012,117130,117186,117195,117338,117348,117450,117469,117569,117723,117740,117919,117987,118161,118182,118313,118385,118507,118537,118545,118679,118989,119032,119157,119387,119506,119695,119739,119764,119827,119990,120071,120138,120163,120310,120348,120476,120566,120584,120638,120719,120823,120925,120955,121043,121379,121405,121485,121552,121666,121679,121902,122006,122039,122271,122322,122331,122358,122483,122508,122529,122620,122675,122895,122896,123001,123061,123256,123534,123581,123601,123663,123926,124159,124182,124294,124321,124443,124632,124754,124828,124833,125017,125043,125072,125094,125284,125301,125438,125582,125586,125595,125838,125849,125874,125904,126064,126107,126127,126414,126578,126670
]

In [ ]:
data[old_rgcs].obs['cell_type']

# Scores

In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden_new.h5ad")

In [ ]:
# A small subpopulation of cells clusters out only within a single time point. 
# This is analytically interesting, but prevents clean comparisons between timepoints for other purposes, so we are going to mask these out for now
data = data[data.obs['cell_type'] != 'injury']

In [ ]:
parthanatos_mask = np.array(data.var['parthanatos'])
apoptosis_mask = np.array(data.var['apoptosis'])
necroptosis_mask = np.array(data.var['necroptosis'])

parthanatos_signature = data.var_names[parthanatos_mask]
apoptosis_signature = data.var_names[apoptosis_mask]
necroptosis_signature = data.var_names[necroptosis_mask]

print(parthanatos_signature)
print(necroptosis_signature)
print(apoptosis_signature)

sc.tl.score_genes(data,parthanatos_signature,score_name="parthanatos")
sc.tl.score_genes(data,necroptosis_signature,score_name="necroptosis")
sc.tl.score_genes(data,apoptosis_signature,score_name="apoptosis")

In [ ]:
# from misc_utils import auto_split_range

# fig,axes = plt.subplots(1,3,figsize=(25,5))
# axes = axes.flatten()

# sc.pl.umap(data,color='parthanatos',ax=axes[0],show=False,**auto_split_range(data.obs['parthanatos']))
# sc.pl.umap(data,color='necroptosis',ax=axes[1],show=False,**auto_split_range(data.obs['necroptosis']))
# sc.pl.umap(data,color='apoptosis',ax=axes[2],show=False,**auto_split_range(data.obs['apoptosis']))

# fig.show()

from misc_utils import auto_split_range

injury_mask = np.array(data.obs['exp_condition'] == 'Mtz')

fig,axes = plt.subplots(2,3,figsize=(25,10))

sc.pl.umap(data[injury_mask],color='parthanatos',ax=axes[0][0],show=False,**auto_split_range(data.obs['parthanatos']))
sc.pl.umap(data[injury_mask],color='necroptosis',ax=axes[0][1],show=False,**auto_split_range(data.obs['necroptosis']))
sc.pl.umap(data[injury_mask],color='apoptosis',ax=axes[0][2],show=False,**auto_split_range(data.obs['apoptosis']))

sc.pl.umap(data[~injury_mask],color='parthanatos',ax=axes[1][0],show=False,**auto_split_range(data.obs['parthanatos']))
sc.pl.umap(data[~injury_mask],color='necroptosis',ax=axes[1][1],show=False,**auto_split_range(data.obs['necroptosis']))
sc.pl.umap(data[~injury_mask],color='apoptosis',ax=axes[1][2],show=False,**auto_split_range(data.obs['apoptosis']))

fig.show()


In [ ]:
fig,axes = plt.subplots(1,2,figsize=(10,5))
sc.pl.dotplot(data[data.obs['exp_condition'] == 'Mtz'],['parthanatos','necroptosis','apoptosis'],groupby="cell_type",ax=axes[0],show=False)
sc.pl.dotplot(data[data.obs['exp_condition'] != 'Mtz'],['parthanatos','necroptosis','apoptosis'],groupby="cell_type",ax=axes[1],show=False)
axes[0].set_title("Injury")
axes[1].set_title("Control")
fig.tight_layout()
fig.show()

# data.obs['time'] = data.obs['time'].astype(str)


In [ ]:
sc.pl.umap(data,color='time')

fig,axes = plt.subplots(2,4,figsize=(20,10))
legend_loc = 'none'
for t,ax in zip([12,24,48,72],axes[0]):
    legend_loc = 'on data' if t == 12 else 'none'
    sub = data[(data.obs['time'] == t) & (data.obs['exp_condition'] == 'Mtz')]
    sc.pl.umap(sub,color='cell_type',legend_loc=legend_loc,ax=ax,show=False)
    ax.set_title(f"Injury {t}h")
for t,ax in zip([12,24,48,72],axes[1]):
    sub = data[(data.obs['time'] == t) & (data.obs['exp_condition'] != 'Mtz')]
    sc.pl.umap(sub,color='cell_type',legend_loc=legend_loc,ax=ax,show=False)
    ax.set_title(f"Control {t}h")
fig.tight_layout()
fig.show()


# Miscellaneous Scratch Work

In [ ]:
# It costs us nothing to include this work, but it is unrelated to the core analysis and is presented a) with no warranty, express or implied, and b) purely for historical purposes

## Manually Sanity-Checking Intra-Cell Type Effects

In [ ]:
# sc.pl.dotplot(data,parthanatos_signature,groupby="cell_type")
rgc_subset = data[data.obs['cell_type'] == 'RGCs',parthanatos_signature]
injury_mask = np.array(rgc_subset.obs['exp_condition'] == 'Mtz')
time_sort_injury = np.argsort(rgc_subset[injury_mask].obs['time'].astype(int))
time_sort_control = np.argsort(rgc_subset[~injury_mask].obs['time'].astype(int))

rgc_matrix = rgc_subset.X.toarray()
injury_matrix = rgc_matrix[injury_mask][time_sort_injury]
control_matrix = rgc_matrix[~injury_mask][time_sort_control]

fig,(ax0,ax1) = plt.subplots(1,2)
fig.suptitle("RGCs only")
ax0.set_title("Injury")
im0 = ax0.imshow(
    injury_matrix,
    cmap='viridis',
    aspect='auto',
    interpolation='nearest',vmin=0,vmax=1.5
)
ax0.set_xticks(np.arange(injury_matrix.shape[1]),labels=parthanatos_signature,rotation=90)

ax1.set_title("Control")
im1 = ax1.imshow(
    control_matrix,
    cmap='viridis',
    aspect='auto',
    interpolation='nearest',vmin=0,vmax=1.5
)
ax1.set_xticks(np.arange(control_matrix.shape[1]),labels=parthanatos_signature,rotation=90)
plt.colorbar(im0,ax=ax0)
plt.colorbar(im1,ax=ax1)
fig.tight_layout()
fig.show()

In [ ]:
directional_exp = np.array(data.obs['exp_condition'].map(lambda x: 1 if "Mtz" in x else -1))
cell_type_one_hot = pd.get_dummies(data.obs['cell_type'])
leiden_one_hot = pd.get_dummies(data.obs['leiden'])

directional_cell_type = [cell_type_one_hot.iloc[:,i].astype(int) * directional_exp for i in range(cell_type_one_hot.shape[1])]
directional_cell_type = np.array(directional_cell_type).T

directional_leiden = [leiden_one_hot.iloc[:,i].astype(int) * directional_exp for i in range(leiden_one_hot.shape[1])]
directional_leiden = np.array(directional_leiden).T

concordance_plot(
    directional_cell_type,
    data.obsm['X_pca'],
    metric='correlation',
    title="Within-group Injury Correlation",
    label_1="Cell Type",
    label_2="PC",
    ticks_1=cell_type_one_hot.columns,
)


concordance_plot(
    directional_leiden,
    data.obsm['X_pca'],
    metric='correlation',
    title="Within-group Injury Correlation",
    label_1="Leiden",
    label_2="PC",
    ticks_1=leiden_one_hot.columns,
    figsize=(15,15)
)


concordance_plot(
    cell_type_one_hot,
    data.obsm['X_pca'],
    ticks_1=cell_type_one_hot.columns,
    title="PCs vs Pure Cell Types",
    label_1="Cell Type",
    label_2="PC",
)



In [ ]:
time_one_hot = pd.get_dummies(data.obs['time'].astype(str))
concordance_plot(
    time_one_hot,
    data.obsm['X_pca'],
    ticks_1=time_one_hot.columns,
    title="PCs vs Time, Categorical",
    label_1="Time",
    label_2="PC",
)




In [ ]:
from misc_utils import auto_split_range

fig,axes = plt.subplots(5,10,figsize=(20,10))
axes = axes.flatten()
for i,ax in enumerate(axes):
    ax.scatter(
        data.obsm['X_umap'][:,0],
        data.obsm['X_umap'][:,1],
        c=data.obsm['X_pca'][:,i],
        **auto_split_range(data.obsm['X_pca'][:,i]),
        s=1,
    )
    ax.set_title(f"PC {i}")
    # data.obs['pc_tmp'] = data.obsm['X_pca'][:,i]
    # sc.pl.umap(data,color='pc_tmp',ax=ax,show=False,**auto_split_range(data.obs['pc_tmp']))
fig.tight_layout()
fig.show()


In [ ]:
# data.obs['exp_tmp'] = np.array(data.obs['exp_condition'].map(lambda x: 1 if "Mtz" in x else 0)).astype(int)
sc.pl.umap(data,color='exp_condition')

## Diverging Heatmap

In [ ]:
n_types = len(data.obs['cell_type'].cat.categories)
n_types

In [ ]:

def diverging_heatmap_order(exp,time):
    exp = [-1 if x=="Mtz" else 1 for x in exp]
    time = [e*t for e,t in zip(exp,time)]
    heatmap_order = np.argsort(np.array(time))
    return heatmap_order


fig,axes = plt.subplots(4,5,figsize=(25,15))
for i,cell_type in enumerate(data.obs['cell_type'].cat.categories):
    print(cell_type)
    ax = axes.flatten()[i]
    subset = data[data.obs['cell_type'] == cell_type]
    heatmap_order = diverging_heatmap_order(subset.obs['exp_condition'],subset.obs['time'])

    local_pcs = subset.obsm['X_pca']
    local_pc_sort = single_agg(local_pcs.T)
    local_pcs = local_pcs[heatmap_order].T[local_pc_sort].T

    ax.imshow(
        local_pcs,
        **auto_split_range(local_pcs)
    )
    ax.set_xticks(np.arange(local_pcs.shape[1]),np.arange(local_pcs.shape[1])[local_pc_sort],rotation=90)
    ax.set_title(cell_type)
fig.tight_layout()
fig.show()

    # plt.figure()
    # plt.imshow(
    #     np.array(pos_subset.X.todense())[heatmap_order],
    #     # **auto_split_range(data.obsm['X_pca'])
    # )
    # plt.show()


## Parallel Heatmap


In [ ]:
# fig,axes = plt.subplots(4,5,figsize=(25,15))

for i,cell_type in enumerate(data.obs['cell_type'].cat.categories):
    # ax = axes.flatten()[i]
    subset = data[data.obs['cell_type'] == cell_type]
    injury_subset = subset[subset.obs['exp_condition'] == 'Mtz']
    control_subset = subset[subset.obs['exp_condition'] != 'Mtz']
    injury_time_sort = np.argsort(injury_subset.obs['time'])
    control_time_sort = np.argsort(control_subset.obs['time'])

    local_pc_sort = single_agg(injury_subset.obsm['X_pca'].T)

    fig,(ax0,ax0_bar,ax1,ax1_bar) = plt.subplots(1,4,width_ratios=(20,1,20,1),figsize=(15,5))
    fig.suptitle(cell_type)
    ax0.imshow(
        injury_subset.obsm['X_pca'][injury_time_sort].T[local_pc_sort].T,
        **auto_split_range(injury_subset.obsm['X_pca'])
    )
    ax0.set_xticks(np.arange(injury_subset.obsm['X_pca'].shape[1]),np.arange(injury_subset.obsm['X_pca'].shape[1])[local_pc_sort],rotation=90)
    ax0_bar.imshow(
        np.array(injury_subset.obs['time'][injury_time_sort]).reshape(-1,1),
    )

    ax1.imshow(
        control_subset.obsm['X_pca'][control_time_sort].T[local_pc_sort].T,
        **auto_split_range(control_subset.obsm['X_pca']),
    )
    ax1.set_xticks(np.arange(control_subset.obsm['X_pca'].shape[1]),np.arange(control_subset.obsm['X_pca'].shape[1])[local_pc_sort],rotation=90)
    ax1_bar.imshow(
        np.array(control_subset.obs['time'][control_time_sort]).reshape(-1,1),
    )

    fig.tight_layout()
    fig.show()

## Other

In [ ]:
leiden_one_hot = pd.get_dummies(data.obs['leiden'])
cell_type_one_hot = pd.get_dummies(data.obs['cell_type'])
exp_one_hot = np.array(data.obs['exp_condition'].map(lambda x: 1 if "Mtz" in x else -1)).reshape(-1,1)
concordance_plot(
    exp_one_hot,
    cell_type_one_hot,
    ticks_2=cell_type_one_hot.columns,
)
concordance_plot(
    exp_one_hot,
    leiden_one_hot,
)

In [ ]:
# # exp = data.obs['exp_condition']
# # time = data.obs['time']
# # exp = [-1 if x=="Mtz" else 1 for x in exp]
# # time = [exp*t for t in time]
# diverging_heatmap_order(data.obs['exp_condition'],data.obs['time'])


In [ ]:
# lol don't do this, just checking what paper did

mg_subset = data[data.obs['cell_type'] == 'Muller glia'].copy()

sc.tl.rank_genes_groups(mg_subset, 'exp_condition')
sc.pl.rank_genes_groups(mg_subset, n_genes=20)







In [ ]:
from scipy.cluster.hierarchy import dendrogram,linkage

def single_agg(mtx,metric='cosine',method='average'):
    return dendrogram(linkage(mtx, metric=metric, method=method), no_plot=True)['leaves']

def double_agg(mtx,metric='cosine',method='average'):
    row_agg = dendrogram(linkage(mtx, metric=metric, method=method), no_plot=True)['leaves']
    col_agg = dendrogram(linkage(mtx.T, metric=metric, method=method), no_plot=True)['leaves']
    return mtx[row_agg].T[col_agg].T


In [ ]:
data.obs['pc_tmp'] = data.obsm['X_pca'][:,4]
injury_mask = data.obs['exp_condition'] == "Mtz"
sc.pl.umap(data[injury_mask],color='pc_tmp',show=False,**auto_split_range(data.obs['pc_tmp']))
sc.pl.umap(data[~injury_mask],color='pc_tmp',show=False,**auto_split_range(data.obs['pc_tmp']))

In [ ]:
injury_mask = data.obs['exp_condition'] == "Mtz"
time_masks = [data.obs['time'] == t for t in [12,24,48,72]]

for i in range(40):
    ct_mask = data.obs['cell_type'] == "Bipolar cells"

    plt.figure()
    plt.title(f"PC {i}")

    plt.violinplot(
        [
            data.obsm['X_pca'][:,i][ct_mask & injury_mask & time_masks[j]]
            for j in range(4)
        ],
        positions=np.arange(4)-.15,widths=0.3
    )        
    plt.violinplot(
        [
            data.obsm['X_pca'][:,i][ct_mask & ~injury_mask & time_masks[j]]
            for j in range(4)
        ],
        positions=np.arange(4)+.15,widths=0.3
    )
    plt.show()



